# Joint RV + Gaia DR4 astrometry — the mass of HD 114762 b

HD 114762 b (Latham et al. 1989) was the **first exoplanet candidate ever**
announced, six years before 51 Peg. Radial velocities give M sin i ~ 11 M_Jup, and
for thirty years it sat between "giant planet" and "brown dwarf" — because **radial
velocity cannot measure inclination**.

Kiefer et al. (2019) settled it with Hipparcos-Gaia astrometry: the orbit is nearly
face-on, i ~ 6 deg, so the true mass is ~0.10 M_sun. The first exoplanet candidate
is a low-mass star.

That makes it the cleanest possible demonstration of why you fit the two channels
together:

| channel | constrains | degenerate in |
|---|---|---|
| radial velocity | K, proportional to M_sec * sin(i) | M_sec and i separately |
| along-scan astrometry | a0, proportional to M_sec; and i | — |

Neither alone gives a mass. Together they do.


In [ ]:
using Nereus
using MCMCChains
using Statistics: median, quantile
using Printf


## 1. Radial velocities

59 velocities over ~29 yr from the California Legacy Survey (Rosenthal et al. 2021):
24 Keck/HIRES and 35 Lick/Hamilton. They ship with Nereus, so nothing to download.

The two instruments are kept as **separate channels**. They have different zero
points and different jitter, and merging them would force one offset and one noise
level onto both.


In [ ]:
rvfile = normpath(joinpath(pathof(Nereus), "..", "..", "test", "data", "hd114762_rv.dat"))

tb = Dict{String, Vector{Float64}}()
for line in eachline(rvfile)
    s = strip(line); (isempty(s) || startswith(s, "#")) && continue
    t, rv, er, ins = split(s)
    d = get!(() -> Float64[], tb, ins)
    append!(d, (parse(Float64,t) - 2_400_000.5, parse(Float64,rv), parse(Float64,er)))
end
mkrv(ins) = let v = reshape(tb[ins], 3, :); (t=v[1,:], rv=v[2,:], rv_err=v[3,:]) end
hires = mkrv("j"); lick = mkrv("lick")

@printf("%d HIRES + %d Lick over %.1f yr\n", length(hires.t), length(lick.t),
        (max(maximum(hires.t),maximum(lick.t)) - min(minimum(hires.t),minimum(lick.t)))/365.25)


## 2. Gaia DR4 epoch astrometry

Same reader as notebook 01 — a different source id.


In [ ]:
const HD114762_SID = 3937211745905473024

xml = get(ENV, "NEREUS_GAIA_DR4_XML", "")
if isempty(xml) || !isfile(xml)
    xml = fetch_gaia_dr4_prerelease()
end
src = read_gaia_epoch_votable(xml, HD114762_SID)
@printf("%d along-scan abscissae, G = %.2f\n", length(src.iad.t), src.g_mag)


## 3. The joint model

`RVAS` mode: one companion constrained by radial velocity **and** absolute
astrometry at once. Nereus builds a single likelihood over both channels — it is not
fitting them separately and combining summaries afterwards.

Per-instrument offsets (`gamma`) and jitter terms come from declaring the two RV
channels; the parallax again carries an informative prior.


In [ ]:
const M_PRI   = 0.83     # M_sun, Kiefer+ 2019
const PLX     = 25.36    # mas, Gaia DR3
const PLX_ERR = 0.30

target = build_target(
    M_pri = M_PRI,
    planets = (b = (
        # TIGHT prior on a, bracketing the 35-yr-established P = 83.92 d.
        # This is a targeted mass/inclination fit, NOT a blind period
        # search — the period has been known since 1989 and re-searching
        # it here would only add multimodality.
        a      = LogUniformPrior(0.30, 0.45),   # AU  => P ~ 60-105 d
        M_sec  = LogUniformPrior(0.003, 0.5),   # M_sun — reaches STELLAR
        sesinw = UniformPrior(-1.0, 1.0),
        secosw = UniformPrior(-1.0, 1.0),
        Mo     = UniformPrior(0.0, 2pi),
        inc    = SinePrior(),
        Omega  = UniformPrior(0.0, 2pi),
    ),),
    rv = (
        HIRES = (data = hires, sigma = LogUniformPrior(0.5, 50.0)),
        Lick  = (data = lick,  sigma = LogUniformPrior(0.5, 50.0)),
    ),
    iad = src.iad,
    plx = NormalPrior(PLX, PLX_ERR),
    M_s = M_PRI,
    # LINEAR trend for the wide outer M-dwarf HD 114762 B. Without it the
    # RVs cannot be fitted. A *quadratic* term over the 29-yr baseline is a
    # near-unconstrained degenerate direction that rails the posterior and
    # breaks the evidence — the right model for B is a second Keplerian.
    trend_order = 1,
)
println("free parameters: ", n_unfrozen(target.params))


Note `M_sec` is allowed up to 0.5 M_sun. A prior that stopped at planetary masses
would make the correct answer unreachable — the fit would rail against the ceiling
and report a planet. Prior ranges are part of the hypothesis being tested.


## 4. Fit

**Use 12 rounds.** This is not a style preference — it is the difference
between a result and a coincidence. Measured on this exact fit:

| | 4 rounds | 12 rounds |
|---|---|---|
| RV chi-squared / point | 58.65 | **0.97** |
| inclination | 0.01 deg (railed) | 4.84 +/- 0.08 deg |
| eccentricity | 0.13 [+0.80, -0.05] | 0.343 [+0.001, -0.002] |
| true mass | 0.113 M_sun | 0.139 M_sun |

Note what the short run does: it rails the inclination against zero, fits the
radial velocities sixty times worse than it should, returns error bars an order
of magnitude too wide — **and still lands within 20 per cent of the right mass.**
A short run that agrees with the answer you were hoping for is the most dangerous
output this code can produce.


In [ ]:
nrounds = parse(Int, get(ENV, "HD114762_ROUNDS", "12"))

t0 = time()
chains, log_Z = sample_pt(target; n_rounds = nrounds, n_chains = 8,
                          seed = 42, show_report = false)
@printf("%d rounds in %.1f min,  log Z = %.2f\n", nrounds, (time()-t0)/60, log_Z)


## 5. The sin(i) break

This is the whole point. `M sin i` is what RV alone would have told you; `M_true` is
what the two channels together give.


In [ ]:
a_v   = vec(Array(chains[:, :a_k1, :]))
M_sec = vec(Array(chains[:, :M_sec_k1, :]))
ses   = vec(Array(chains[:, :sesinw_k1, :]))
sec   = vec(Array(chains[:, :secosw_k1, :]))
inc_v = vec(Array(chains[:, :inc_k1, :]))
e_v    = ses.^2 .+ sec.^2
P_d    = [365.25*sqrt(a_v[j]^3/(M_PRI+M_sec[j])) for j in eachindex(a_v)]
Msini  = M_sec .* sin.(inc_v) .* 1047.57      # M_Jup
Mtrue  = M_sec                                 # M_sun
band(x)=(quantile(x,0.5),quantile(x,0.84)-quantile(x,0.5),quantile(x,0.5)-quantile(x,0.16))
println("                    median [+1sig, -1sig]        Kiefer+ 2019")
let (m,hi,lo)=band(P_d);   @printf("  P (d)       %8.2f [+%.2f, -%.2f]     83.92\n",m,hi,lo) end
let (m,hi,lo)=band(e_v);   @printf("  e           %8.3f [+%.3f, -%.3f]     0.335\n",m,hi,lo) end
let (m,hi,lo)=band(Msini); @printf("  M sin i(MJ) %8.2f [+%.2f, -%.2f]     ~11 (RV-only)\n",m,hi,lo) end
let (m,hi,lo)=band(rad2deg.(inc_v)); @printf("  i (deg)     %8.2f [+%.2f, -%.2f]     6.2\n",m,hi,lo) end
let (m,hi,lo)=band(Mtrue); @printf("  M_true(Msun)%8.3f [+%.3f, -%.3f]     0.103  <- A STAR\n",m,hi,lo) end


## 6. Plots

`plot_rv_astrom_phasefold` is the one that carries the argument: the RV curve and
the astrometric orbit on the same posterior.


In [ ]:
outdir = joinpath(pwd(), "hd114762_figures")
mkpath(outdir)

plot_rv_timeseries(chains, target.params, target.data; output = outdir)
plot_rv_phasefold(chains, target.params, target.data; output = outdir)
plot_rv_astrom_phasefold(chains, target.params, target.data; output = outdir)
plot_iad_residuals(chains, target.params, target.data; output = outdir)
plot_orbit_skyplane(chains, target.params, target.data; output = outdir)
plot_corner(chains, target.params; output = outdir)


# Nereus files model figures under a models/ subdirectory.
for (root, _, files) in walkdir(outdir), fl in files
    println("  ", relpath(joinpath(root, fl), outdir))
end


In [ ]:
save_chains(joinpath(outdir, "chains.nc"), chains, target.params;
            data = target.data, log_evidence = log_Z)
println("saved -> ", joinpath(outdir, "chains.nc"))


---

## What to take away

1. RV measures `M sin i`. It cannot measure a mass, ever, on its own.
2. Absolute astrometry measures the angular wobble, which needs a distance to become
   a mass — hence the informative parallax prior.
3. Fit jointly and the degeneracy breaks. Here it turns an 11 M_Jup "planet" into a
   0.10 M_sun star.
4. This is what Gaia DR4 makes routine: per-CCD abscissae for a hundred million
   sources, rather than five fitted parameters per star.
